In [1]:
import pandas as pd

BAT_RESULTS_PATH = "bat_posts_results_final_patched.csv"
MASTER_PATH       = "/Users/nadia/Desktop/redditRun_june/master_posts_classified.csv"   # has 'subreddit' and other metadata
OUTPUT_PATH        = "bat_posts_results_with_subreddit.csv"

bat_df = pd.read_csv(BAT_RESULTS_PATH)
master_df = pd.read_csv(MASTER_PATH)

print(f"BAT results : {len(bat_df)} rows, columns: {bat_df.columns.tolist()}")
print(f"Master file : {len(master_df)} rows, columns: {master_df.columns.tolist()}")

bat_df["post_id"] = bat_df["post_id"].astype(str).str.strip()
master_df["id"] = master_df["id"].astype(str).str.strip()

# Pull in subreddit + a few other useful metadata columns, avoid duplicating
# columns that already exist in bat_df (like 'text')
metadata_cols = ["id", "subreddit", "author", "created_date", "score",
                  "num_comments", "permalink", "url"]
metadata_cols = [c for c in metadata_cols if c in master_df.columns]

merged = bat_df.merge(
    master_df[metadata_cols],
    left_on="post_id",
    right_on="id",
    how="left"
)
merged = merged.drop(columns=["id"])  # redundant with post_id after merge

n_missing = merged["subreddit"].isna().sum()
print(f"\nRows with subreddit successfully matched : {len(merged) - n_missing}/{len(merged)}")
if n_missing:
    print(f"  [!] {n_missing} rows had no matching post_id in master file — check for ID mismatches")

merged.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved: {OUTPUT_PATH}")

print("\nSubreddit distribution:")
print(merged["subreddit"].value_counts())

BAT results : 144652 rows, columns: ['row_type', 'post_id', 'comment_id', 'text', 'triage', 'na_subtype', 'triage_reason', 'EX', 'EMO', 'COG', 'MD', 'bat_score', 'EX_reasoning', 'EMO_reasoning', 'COG_reasoning', 'MD_reasoning']
Master file : 366115 rows, columns: ['id', 'subreddit', 'author', 'created_date', 'created_utc', 'year', 'month', 'day_of_week', 'hour', 'year_month', 'title', 'selftext', 'text', 'cleaned_text', 'original_text_len', 'cleaned_text_len', 'word_count', 'score', 'upvote_ratio', 'num_comments', 'link_flair_text', 'permalink', 'url', 'text_processed', 'predicted_label', 'prediction_confidence']

Rows with subreddit successfully matched : 144652/144652

Saved: bat_posts_results_with_subreddit.csv

Subreddit distribution:
subreddit
sysadmin                109814
cybersecurity            23965
SecurityCareerAdvice      5647
AskNetsec                 5058
ciso                       168
Name: count, dtype: int64


In [2]:
import pandas as pd

INPUT_PATH = "bat_posts_results_with_subreddit.csv"   # <-- edit path if needed

df = pd.read_csv(INPUT_PATH)
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")

Shape: 144652 rows x 23 columns


In [3]:
# ── Headers / columns ──────────────────────────────────────────────────────
print("Columns:")
for c in df.columns:
    print(f"  - {c} ({df[c].dtype})")

Columns:
  - row_type (object)
  - post_id (object)
  - comment_id (float64)
  - text (object)
  - triage (float64)
  - na_subtype (float64)
  - triage_reason (float64)
  - EX (object)
  - EMO (object)
  - COG (object)
  - MD (object)
  - bat_score (int64)
  - EX_reasoning (object)
  - EMO_reasoning (object)
  - COG_reasoning (object)
  - MD_reasoning (object)
  - subreddit (object)
  - author (object)
  - created_date (object)
  - score (float64)
  - num_comments (float64)
  - permalink (object)
  - url (object)


In [4]:
# ── bat_score: 0 vs >0 ────────────────────────────────────────────────────
n_zero = (df["bat_score"] == 0).sum()
n_positive = (df["bat_score"] > 0).sum()

print(f"bat_score == 0  : {n_zero} ({n_zero/len(df)*100:.1f}%)")
print(f"bat_score >  0  : {n_positive} ({n_positive/len(df)*100:.1f}%)")
print(f"Total           : {len(df)}")

print("\nFull bat_score distribution (0-4):")
print(df["bat_score"].value_counts().sort_index())

bat_score == 0  : 132415 (91.5%)
bat_score >  0  : 12237 (8.5%)
Total           : 144652

Full bat_score distribution (0-4):
bat_score
0    132415
1      7232
2      3043
3      1573
4       389
Name: count, dtype: int64


In [5]:
# ── Subreddit distribution where bat_score > 0 ──────────────────────────────
positive_df = df[df["bat_score"] > 0]

print(f"\nSubreddit distribution where bat_score > 0 (n={len(positive_df)}):")
print(positive_df["subreddit"].value_counts())

print(f"\nSubreddit distribution where bat_score > 0 (%):")
print((positive_df["subreddit"].value_counts(normalize=True) * 100).round(1))


Subreddit distribution where bat_score > 0 (n=12237):
subreddit
sysadmin                9871
cybersecurity           1626
SecurityCareerAdvice     529
AskNetsec                199
ciso                      12
Name: count, dtype: int64

Subreddit distribution where bat_score > 0 (%):
subreddit
sysadmin                80.7
cybersecurity           13.3
SecurityCareerAdvice     4.3
AskNetsec                1.6
ciso                     0.1
Name: proportion, dtype: float64


In [7]:
import pandas as pd

INPUT_PATH = "/Users/nadia/Desktop/redditRun_june/comment_data/bat_score_pos.csv"   # <-- edit path if needed

df = pd.read_csv(INPUT_PATH)
df.head(5)

,row_type,post_id,comment_id,text,triage,na_subtype,triage_reason,EX,EMO,COG,MD,bat_score,EX_reasoning,EMO_reasoning,COG_reasoning,MD_reasoning
0,post,12t87xb,NaN,Am I the Only One...\nAm I the only one who ge...,NaN,NaN,NaN,NO,NO,NO,YES,1,"No mention of physical tiredness, mental deple...","No intense emotional reactions, irritability, ...","No indication of brain fog, memory issues, dif...","""jaded me"" and ""Is that all, really?"" indicati..."
1,post,1epl8ed,NaN,Advice for Head of Infosec\nI have 10 years of...,NaN,NaN,NaN,YES,NO,NO,NO,1,prolonged stress,The post describes a stressful situation and e...,"No cognitive symptoms such as brain fog, forge...","While planning to resign, the author does not ..."
2,post,1fbh2gy,NaN,Time for a change\nAnonymous account.\n\nI cur...,NaN,NaN,NaN,NO,YES,NO,NO,1,While the post describes sustained difficulty ...,incredibly frustrated; Frustrated that every d...,The post describes organizational dysfunction ...,The author explicitly states 'I enjoy the role...
3,post,1k7d8py,NaN,Burnout - How to leave cyber security entirely...,NaN,NaN,NaN,YES,YES,NO,YES,3,"""So burned out"", ""So tired of the constant bat...","""tired of the constant battles"", ""atrocious"", ...",While the author expresses career indecision a...,"""I am just... done"", ""thinking of leaving info..."
4,post,1mzmorn,NaN,"CISO with no team, IT wants “IT security” - ad...",NaN,NaN,NaN,YES,YES,NO,NO,2,"""this setup feels unsustainable"", ""responsibil...","""this situation is the last straw"" and ""I feel...","No evidence of memory problems, brain fog, or ...",Poster remains actively engaged in solving the...
